# EDA — Dataset regenerado (10,000 clientes)

Análisis exploratorio de `data/dataset_regenerado_10000_clientes.csv`, generado en `dataset_regeneration.ipynb`
a partir de las políticas de regeneración y del árbol de decisión de segmento y tarjeta.

> ⚠️ Este EDA fue elaborado por una **IA (Claude, de Anthropic)**. Los datos son sintéticos: lo que se ve aquí refleja las políticas y supuestos con los que se generaron, no clientes reales.

**Contenido**
1. Setup y carga
2. Calidad de datos
3. Distribuciones
4. Segmentos y tarjetas
5. Relaciones entre variables
6. Perfil por segmento y por tarjeta
7. Validación de políticas
8. Bolsas de oportunidad
9. Conclusiones

## 1. Setup y carga

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

# Render nativo de plotly (VS Code / JupyterLab)
pio.renderers.default = "plotly_mimetype"

# Misma paleta y tema que EDA_segmentos.ipynb
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
GRIS = "#a3a29c"
POS, NEG, NEUTRO = "#2a78d6", "#e34948", "#f0efec"
DIVERGENTE = [[0, "#b83a3a"], [0.25, "#e88a89"], [0.5, NEUTRO], [0.75, "#86b6ef"], [1, "#1c5cab"]]

pio.templates["santander"] = go.layout.Template(
    layout=dict(
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color="#0b0b0b"),
        paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb",
        colorway=PAL,
        xaxis=dict(gridcolor="#e8e7e3", zerolinecolor="#c3c2b7", linecolor="#c3c2b7"),
        yaxis=dict(gridcolor="#e8e7e3", zerolinecolor="#c3c2b7", linecolor="#c3c2b7"),
        hoverlabel=dict(bgcolor="white", font_size=12),
        margin=dict(l=60, r=30, t=60, b=50),
        bargap=0.25,
    )
)
pio.templates.default = "simple_white+santander"

DATA = "data/dataset_regenerado_10000_clientes.csv"
clientes = pd.read_csv(DATA)

SEGMENTOS = ["Mass", "Select", "Premier"]
COLOR_SEG = dict(zip(SEGMENTOS, PAL))  # el color sigue al segmento en todas las gráficas
TARJETAS = ["Free", "LikeU", "Aeromexico Blanca", "Fiesta Rewards Oro", "Santander Gold",   # de menor a mayor tier
            "Santander Platinum", "Fiesta Rewards Platinum", "Aeromexico Platinum", "World Elite"]
NUM = clientes.select_dtypes("number").columns.tolist()

print("clientes:", clientes.shape)
display(clientes.head())

## 2. Calidad de datos

In [ ]:
calidad = pd.DataFrame({
    "dtype": clientes.dtypes.astype(str),
    "nulos": clientes.isna().sum(),
    "unicos": clientes.nunique(),
    "min": clientes.min(numeric_only=True),
    "max": clientes.max(numeric_only=True),
})
print("Duplicados de fila:", clientes.duplicated().sum(),
      "| customer_id duplicados:", clientes.customer_id.duplicated().sum())
display(calidad)

rangos = {
    "uso_tarjeta_pct": (0, 100), "ecommerce_pct": (0, 100), "pagos_puntuales_pct": (0, 100),
    "activacion_promos_pct": (0, 100), "abandono_riesgo": (0, 1), "nps": (0, 10), "satisfaccion": (1, 5),
}
fuera = {c: int((~clientes[c].between(*r)).sum()) for c, r in rangos.items()}
print("Registros fuera de rango:", fuera)
print("Clientes menores de 18 años:", int((clientes.edad < 18).sum()))
display(clientes[NUM].describe(percentiles=[.05, .25, .5, .75, .95]).T)

## 3. Distribuciones

In [ ]:
fig = make_subplots(rows=4, cols=4, subplot_titles=NUM, vertical_spacing=0.08, horizontal_spacing=0.05)
for i, v in enumerate(NUM):
    fig.add_histogram(x=clientes[v], nbinsx=40, marker_color=PAL[0], showlegend=False,
                      hovertemplate=v + ": %{x}<br>clientes: %{y}<extra></extra>", row=i // 4 + 1, col=i % 4 + 1)
fig.update_layout(height=900, title="Distribución de cada variable numérica", bargap=0.05)
fig.show()

In [ ]:
# Pirámide de edad: muestra vs pesos de la política
POBLACION = pd.Series([22_244_425, 21_105_726, 18_729_814, 15_640_829, 12_087_127, 7_384_543, 4_326_566],
                      index=["15-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"])
rango = pd.cut(clientes.edad, [14, 24, 34, 44, 54, 64, 74, 200], labels=POBLACION.index)
edades = pd.DataFrame({"muestra %": rango.value_counts(normalize=True).sort_index() * 100,
                       "política %": POBLACION / POBLACION.sum() * 100})
display(edades)
fig = go.Figure()
fig.add_bar(x=edades.index, y=edades["muestra %"], name="Muestra", marker_color=PAL[0])
fig.add_scatter(x=edades.index, y=edades["política %"], name="Política (pirámide 2026)", mode="markers",
                marker=dict(color="#0b0b0b", size=10, symbol="diamond"))
fig.update_layout(height=360, title="Edad: la muestra replica la pirámide poblacional", yaxis_title="% de clientes",
                  legend=dict(orientation="h", y=1.12, x=1, xanchor="right"))
fig.show()

**Lectura — distribuciones**

- Base limpia por construcción: 10,000 clientes únicos, 0 nulos, 0 duplicados y 0 valores fuera de rango.
- La **edad** replica la pirámide 2026 (diferencias < 1 pp por rango). Ojo: **637 clientes (6.4 %) tienen 15–17 años**, algo que la política permite pero que en la vida real no ocurre con tarjetas de crédito.
- **Ingreso y gasto** tienen cola derecha (log-normal): mediana de $10.7k y $8.9k, máximos de $149k y $108k. Para resumirlos conviene usar la mediana.
- `antiguedad_meses` llega a 881 meses (73 años) porque es uniforme entre 1 y (edad − 15)·12; la mediana es de 109 meses.
- `ecommerce_pct` se **satura en 100 %** en la parte alta (p95 = 100) por el tope de la fórmula.
- `nps` (media 6.9) y `satisfaccion` (media 4.0) salen exactamente con las medias de la política.

## 4. Segmentos y tarjetas

In [ ]:
ct = pd.crosstab(clientes.tarjeta_santander, clientes.segmento).reindex(index=TARJETAS, columns=SEGMENTOS)
display(ct.assign(Total=ct.sum(axis=1), **{"% base": ct.sum(axis=1) / len(clientes) * 100}))

fig = go.Figure()
for s in SEGMENTOS:
    fig.add_bar(y=ct.index, x=ct[s], name=s, orientation="h", marker_color=COLOR_SEG[s],
                marker_line=dict(color="#fcfcfb", width=2),
                hovertemplate="%{y} · " + s + "<br>%{x:,} clientes<extra></extra>")
fig.update_layout(barmode="stack", height=420, yaxis_autorange="reversed", xaxis_title="clientes",
                  title="Clientes por tarjeta (de menor a mayor tier) y segmento",
                  legend=dict(orientation="h", y=1.1, x=1, xanchor="right"))
fig.show()

**Lectura — segmentos y tarjetas**

- Mezcla: **Mass 48 %, Select 42 %, Premier 10 %**.
- Cada tarjeta pertenece a un solo segmento, excepto las tres **Platinum**, que se reparten casi a partes iguales entre Select y Premier, como dice la política.
- **Santander Gold** es la tarjeta más común (32 %): en Select concentra a quien gasta menos que la media *y* a quien tiene poca antigüedad (el hueco de la política se llenó con Gold).
- **World Elite** queda en 85 clientes (0.85 %): un nicho, como corresponde a "mayores ingresos, antigüedad y gastos".

## 5. Relaciones entre variables

In [ ]:
corr = clientes[NUM].corr(method="spearman")
fig = px.imshow(corr, color_continuous_scale=DIVERGENTE, zmin=-1, zmax=1, text_auto=".2f", aspect="auto",
                title="Correlación de Spearman entre variables")
fig.update_layout(height=620, coloraxis_colorbar=dict(title="ρ"))
fig.show()

In [ ]:
# La regla de segmento (árbol 1) vista en los datos: gasto vs antigüedad
muestra = clientes.sample(3_000, random_state=0)
fig = go.Figure()
for s in SEGMENTOS:
    d = muestra[muestra.segmento == s]
    fig.add_scatter(x=d.antiguedad_meses, y=d.gasto_mensual, mode="markers", name=s,
                    marker=dict(color=COLOR_SEG[s], size=8, opacity=0.6, line=dict(color="#fcfcfb", width=1)),
                    customdata=d.customer_id, hovertemplate="%{customdata}<br>antigüedad %{x} m<br>gasto $%{y:,.0f}<extra>" + s + "</extra>")
for y, x, t in [(8_000, 12, "Select: gasto ≥ 8k y antigüedad ≥ 12 m"), (20_000, 36, "Premier: gasto ≥ 20k y antigüedad ≥ 36 m")]:
    fig.add_shape(type="line", x0=x, x1=900, y0=y, y1=y, line=dict(color="#52514e", dash="dot", width=1.5))
    fig.add_shape(type="line", x0=x, x1=x, y0=y, y1=150_000, line=dict(color="#52514e", dash="dot", width=1.5))
    fig.add_annotation(x=np.log10(900), y=np.log10(y), text=t, showarrow=False, xanchor="right", yanchor="bottom",
                       font=dict(size=11, color="#52514e"))
fig.update_xaxes(type="log", title="antigüedad (meses, log)")
fig.update_yaxes(type="log", title="gasto mensual (MXN, log)")
fig.update_layout(height=520, title="Gasto vs antigüedad por segmento (muestra de 3,000; líneas = reglas del árbol)",
                  legend=dict(orientation="h", y=1.08, x=1, xanchor="right"))
fig.show()

In [ ]:
# Relaciones de la política que no son lineales en la matriz: e-commerce por edad y abandono por antigüedad
grupos = pd.DataFrame({
    "edad": pd.cut(clientes.edad, [14, 24, 34, 44, 54, 64, 74, 200], labels=POBLACION.index),
    "antig": pd.cut(clientes.antiguedad_meses, [0, 6, 12, 24, 36, 120, 1_000],
                    labels=["≤6", "7-12", "13-24", "25-36", "37-120", ">120"]),
})
fig = make_subplots(rows=1, cols=2, subplot_titles=["E-commerce % (mediana) por edad", "Riesgo de abandono (mediana) por antigüedad (meses)"])
for col, (g, v) in enumerate([("edad", "ecommerce_pct"), ("antig", "abandono_riesgo")], start=1):
    m = clientes.groupby(grupos[g], observed=True)[v].median()
    fig.add_scatter(x=m.index.astype(str), y=m.values, mode="lines+markers", line=dict(color=PAL[0], width=2),
                    marker=dict(size=8), showlegend=False, hovertemplate="%{x}<br>%{y:.2f}<extra></extra>", row=1, col=col)
fig.update_layout(height=360)
fig.show()

**Lectura — relaciones**

- Al revés que el dataset original (|ρ| ≈ 0.03), aquí **todo el bloque de consumo está muy correlacionado**: ingreso, gasto, transacciones, uso, e-commerce y logins tienen ρ entre 0.84 y 0.98, porque las fórmulas los encadenan. Para modelar habrá que cuidar la **multicolinealidad** (por ejemplo, quedarse con una sola variable del bloque o usar PCA).
- `activacion_promos_pct` es la única variable de comportamiento que va **en contra** del ingreso (ρ ≈ −0.55), sobre todo por el nivel base de cada segmento.
- `nps` y `satisfaccion` no se relacionan con nada salvo con `abandono_riesgo`: se generaron independientes.
- El gráfico gasto vs antigüedad muestra los **rectángulos exactos** del árbol de segmento: no hay ruido en la frontera.
- El e-commerce baja con la edad y el abandono cae con la antigüedad hasta 36 meses y luego se aplana (por el tope de la fórmula).

## 6. Perfil por segmento y por tarjeta

In [ ]:
VARS_PERFIL = ["edad", "antiguedad_meses", "ingreso_mensual", "gasto_mensual", "uso_tarjeta_pct", "transacciones_mes",
               "ecommerce_pct", "pagos_puntuales_pct", "app_logins_mes", "nps", "satisfaccion",
               "abandono_riesgo", "activacion_promos_pct"]
display(clientes.groupby("segmento")[VARS_PERFIL].median().reindex(SEGMENTOS).T.rename_axis("mediana"))

CLAVE = ["ingreso_mensual", "gasto_mensual", "uso_tarjeta_pct", "ecommerce_pct", "abandono_riesgo", "activacion_promos_pct"]
fig = make_subplots(rows=1, cols=len(CLAVE), subplot_titles=CLAVE, horizontal_spacing=0.04)
for i, v in enumerate(CLAVE, start=1):
    for s in SEGMENTOS:
        fig.add_box(y=clientes.loc[clientes.segmento == s, v], name=s, marker_color=COLOR_SEG[s], boxmean=True,
                    legendgroup=s, showlegend=(i == 1), row=1, col=i)
fig.update_xaxes(showticklabels=False)
fig.update_layout(height=400, title="Métricas clave por segmento (◇ = media)",
                  legend=dict(orientation="h", y=1.18, x=1, xanchor="right"))
fig.show()

In [ ]:
med = clientes.groupby("tarjeta_santander")[VARS_PERFIL].median().reindex(TARJETAS)
z = (med - med.mean()) / med.std()   # color = qué tan arriba/abajo está la tarjeta vs las demás
fig = go.Figure(go.Heatmap(z=z.values, x=VARS_PERFIL, y=TARJETAS, colorscale=DIVERGENTE, zmid=0, zmin=-2.5, zmax=2.5,
                           text=med.map(lambda x: f"{x:,.0f}" if abs(x) >= 100 else f"{x:.2f}").values,
                           texttemplate="%{text}", textfont=dict(size=10),
                           hovertemplate="%{y}<br>%{x}: %{text}<extra></extra>", colorbar=dict(title="z")))
fig.update_layout(height=480, yaxis_autorange="reversed", xaxis_tickangle=-35,
                  title="Mediana por tarjeta (texto) y posición relativa entre tarjetas (color, z-score)")
fig.show()

**Lectura — perfiles**

- Por segmento, el ingreso mediano casi se duplica de un escalón al siguiente (Mass $6.7k → Select $13.9k → Premier $28.6k), y lo mismo pasa con el gasto, las transacciones y los logins.
- Premier tiene un **uso de tarjeta del 93 %** y el **67 % de sus clientes está en 100 % de e-commerce** (saturación).
- La **activación de promociones baja con el segmento** (44 → 35 → 25 %), como pide la política.
- Por tarjeta, la escala de tiers se ve clara: Free < LikeU < Blanca / Oro ≈ Gold < Platinums < World Elite.
- Hallazgo: **Aeroméxico Blanca y Fiesta Rewards Oro tienen el mayor riesgo de abandono** (mediana 0.30–0.35) y la menor antigüedad (16–25 meses). Son clientes con ingreso alto que quedaron en Mass por ser nuevos, y el abandono depende mucho de la antigüedad.

## 7. Validación de políticas

Se comprueba que los datos cumplan (a) las reglas duras de la generación y (b) el **sentido** de cada relación que pide la política.

In [ ]:
c = clientes
seg_regla = np.select(
    [(c.gasto_mensual >= 20_000) & (c.antiguedad_meses >= 36), (c.gasto_mensual >= 8_000) & (c.antiguedad_meses >= 12)],
    ["Premier", "Select"], "Mass")  # umbrales de U en dataset_regeneration.ipynb
reglas = pd.Series({
    "antigüedad ≤ (edad−15)·12 + 1": (c.antiguedad_meses <= (c.edad - 15) * 12 + 1).mean(),
    "gasto ≤ 1.1 × ingreso": (c.gasto_mensual <= 1.1 * c.ingreso_mensual + 10).mean(),
    "uso de tarjeta en (0, 100)": c.uso_tarjeta_pct.between(0, 100, inclusive="neither").mean(),
    "segmento = regla del árbol 1": (c.segmento == seg_regla).mean(),
}, name="% de clientes que cumple") * 100
display(reglas.to_frame())

# Spearman = Pearson sobre rangos (Series.corr(method="spearman") pide scipy)
holgura = (c.ingreso_mensual - c.gasto_mensual) / c.ingreso_mensual
ESPERADO = [  # (variable, contra, signo esperado según la política)
    ("gasto_mensual", c.ingreso_mensual, "ingreso", +1),
    ("pagos_puntuales_pct", holgura, "holgura (ingreso−gasto)/ingreso", +1),
    ("ecommerce_pct", c.gasto_mensual, "gasto", +1),
    ("ecommerce_pct", c.edad, "edad", -1),
    ("transacciones_mes", c.gasto_mensual, "gasto", +1),
    ("app_logins_mes", c.ecommerce_pct, "ecommerce", +1),
    ("app_logins_mes", c.transacciones_mes, "transacciones", +1),
    ("abandono_riesgo", c.satisfaccion, "satisfacción", -1),
    ("abandono_riesgo", c.nps, "nps", -1),
    ("abandono_riesgo", c.antiguedad_meses.clip(upper=36), "antigüedad (tope 36 m)", -1),
    ("uso_tarjeta_pct", c.ingreso_mensual, "ingreso", +1),
    ("uso_tarjeta_pct", c.transacciones_mes, "transacciones", +1),
]
signos = pd.DataFrame([{"variable": v, "contra": nombre, "esperado": "+" if s > 0 else "−",
                        "ρ Spearman": c[v].rank().corr(x.rank())} for v, x, nombre, s in ESPERADO])
# activación de promos: dentro de cada segmento debe bajar con el ingreso
for seg in SEGMENTOS:
    d = c[c.segmento == seg]
    signos.loc[len(signos)] = ["activacion_promos_pct", f"ingreso (dentro de {seg})", "−",
                               d.activacion_promos_pct.rank().corr(d.ingreso_mensual.rank())]
signos["cumple"] = np.where(np.sign(signos["ρ Spearman"]) == signos.esperado.map({"+": 1, "−": -1}), "✓", "✗")
display(signos)
assert (signos.cumple == "✓").all(), "Alguna relación no tiene el sentido que pide la política"

**Lectura — validación**

- Las 4 reglas duras se cumplen en el **100 %** de los clientes, incluida la reconstrucción del segmento con las reglas del árbol 1.
- Las **15 relaciones** de la política tienen el signo correcto.
- Las más débiles son e-commerce vs edad (ρ = −0.11) y activación vs ingreso dentro del segmento (ρ entre −0.06 y −0.13): van en el sentido correcto, pero el efecto es pequeño frente al ruido. Si se busca un efecto más visible, hay que subir los coeficientes en `dataset_regeneration.ipynb`.

## 8. Bolsas de oportunidad

Al ser datos sintéticos, los cortes se definen por percentil de la base (no hay umbrales de negocio reales).

In [ ]:
q = clientes.quantile(numeric_only=True, q=[.25, .75, .9])
FLAGS = {
    "Bajo uso de tarjeta (p25)": clientes.uso_tarjeta_pct <= q.loc[.25, "uso_tarjeta_pct"],
    "Baja activación de promos (p25)": clientes.activacion_promos_pct <= q.loc[.25, "activacion_promos_pct"],
    "Alto riesgo de abandono (p90)": clientes.abandono_riesgo >= q.loc[.9, "abandono_riesgo"],
    "Detractores NPS (0–6)": clientes.nps <= 6,
    "Baja adopción digital (p25 logins)": clientes.app_logins_mes <= q.loc[.25, "app_logins_mes"],
    "Alto gasto (p75)": clientes.gasto_mensual >= q.loc[.75, "gasto_mensual"],
}
FLAGS["Retención: alto riesgo + detractor"] = FLAGS["Alto riesgo de abandono (p90)"] & FLAGS["Detractores NPS (0–6)"]
bolsas = pd.DataFrame({k: f.groupby(clientes.segmento).mean().reindex(SEGMENTOS) * 100 for k, f in FLAGS.items()}).T
bolsas["Base"] = [f.mean() * 100 for f in FLAGS.values()]
bolsas["clientes"] = [int(f.sum()) for f in FLAGS.values()]
display(bolsas.rename_axis("% de clientes del segmento"))

fig = go.Figure()
for s in SEGMENTOS:
    fig.add_bar(y=bolsas.index, x=bolsas[s], name=s, orientation="h", marker_color=COLOR_SEG[s],
                hovertemplate="%{y}<br>" + s + ": %{x:.1f}%<extra></extra>")
fig.update_layout(barmode="group", height=480, yaxis_autorange="reversed", xaxis_title="% de clientes del segmento",
                  title="Bolsas de oportunidad por segmento", legend=dict(orientation="h", y=1.08, x=1, xanchor="right"))
fig.show()

**Lectura — bolsas**

- Las bolsas están **casi separadas por segmento**, una consecuencia de que todo depende del ingreso:
  - **Mass** concentra el bajo uso (50 %), la baja adopción digital (54 %) y el riesgo de abandono alto (19 %).
  - **Premier** concentra la baja activación de promociones (79 %) y todo el alto gasto.
- Los **detractores** (~39 %) se reparten igual en todos los segmentos, porque el NPS es independiente del resto.
- **Retención prioritaria:** 507 clientes con riesgo alto y que son detractores, casi todos en Mass.

## 9. Conclusiones

> 🤖 Conclusiones redactadas por una **IA (Claude, de Anthropic)** a partir de los resultados de arriba.

1. **El dataset regenerado cumple sus políticas.** Hay 0 errores de calidad, las reglas duras se cumplen al 100 % y las 15 relaciones tienen el sentido esperado. A diferencia del dataset original, aquí sí hay perfiles distintos por segmento y por tarjeta.
2. **Todo gira alrededor del ingreso.** Gasto, transacciones, uso, e-commerce y logins son casi la misma variable (ρ > 0.84). Es útil para la segmentación, pero cualquier modelo debe manejar la multicolinealidad.
3. **Segmentos limpios, quizá demasiado.** Las fronteras son reglas exactas y las bolsas de oportunidad caen casi enteras en un solo segmento. Para un caso más realista se podría añadir ruido a la asignación (por ejemplo, un 5–10 % de excepciones).
4. **Oportunidades que surgen de los datos:**
   - Activación y adopción digital en **Mass** (bajo uso y pocos logins).
   - Promociones de beneficio, no de precio, en **Premier**, porque casi no activa promociones.
   - Retención de los clientes nuevos de **Aeroméxico Blanca / Fiesta Oro**, que tienen el mayor riesgo de abandono.
5. **Pendientes en la generación:**
   - Reemplazar las medias de ingreso supuestas por las de DataMexico.
   - Decidir si se aceptan clientes de 15–17 años.
   - Revisar la saturación de e-commerce en 100 %.
   - Si se quiere un efecto más visible, subir los coeficientes de edad (en e-commerce) y de ingreso (en activación).